# Kaggle Deployment: Qwen3-8B Manimator GRPO -> Merged Safetensors & Q8_0 GGUF

This self-contained notebook performs:
1. **Environment Setup & Hardware Detection** (GPU, CUDA, RAM, Disk, apt/pip dependencies, build llama.cpp)
2. **Download Models** (`Qwen/Qwen3-8B` & `nabin2004/qwen-Manimator-1-grpo-clean`)
3. **Merge LoRA Adapter** into base model weights
4. **Deploy Merged Safetensors Model** to Hugging Face (`nabin2004/qwen-Manimator-1-grpo-merged`)
5. **Convert Merged Model to GGUF (F16)**
6. **Quantize to Q8_0** (`qwen-Manimator-1-grpo-Q8_0.gguf`)
7. **Validate GGUF** architecture and baseline generation
8. **Test 32K Context Configuration** (`context = 32768`)
9. **Run End-to-End Manim Generation & Video Compilation Test**
10. **Upload Q8_0 GGUF to Hugging Face Hub** (`nabin2004/qwen-Manimator-1-grpo-GGUF`)
11. **Verify Uploads & Print Final Summary Report**

> **Prerequisite**: Add your Hugging Face write token under **Add-ons -> Secrets** with label `HF_TOKEN`.

In [ ]:
# 01_environment_setup
import os, sys, shutil, subprocess, psutil
from pathlib import Path

WORK_DIR = Path("/kaggle/working") if Path("/kaggle").is_dir() else Path("./kaggle_work").resolve()
WORK_DIR.mkdir(parents=True, exist_ok=True)
LLAMA_CPP_DIR = WORK_DIR / "llama.cpp"

# 1. Hardware Detection
import torch
has_cuda = torch.cuda.is_available()
gpu_name = torch.cuda.get_device_name(0) if has_cuda else "None (CPU only)"
cuda_ver = torch.version.cuda if has_cuda else "N/A"
gpu_mem = f"{torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB" if has_cuda else "0 GB"
cpu_count = psutil.cpu_count(logical=True)
ram_gb = f"{psutil.virtual_memory().total / 1e9:.2f} GB"
disk_free = f"{shutil.disk_usage(WORK_DIR).free / 1e9:.2f} GB"

print("="*50)
print(f"GPU:        {gpu_name}")
print(f"CUDA:       {cuda_ver}")
print(f"GPU memory: {gpu_mem}")
print(f"CPU:        {cpu_count} vCPUs")
print(f"RAM:        {ram_gb}")
print(f"Disk:       {disk_free} free in {WORK_DIR}")
print("="*50)

# 2. Authenticate HF
hf_token = os.environ.get("HF_TOKEN")
if not hf_token:
    try:
        from kaggle_secrets import UserSecretsClient
        hf_token = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        hf_token = None
if not hf_token:
    raise RuntimeError("HF_TOKEN not found! Please attach Kaggle Secret 'HF_TOKEN' with write permissions.")
os.environ["HF_TOKEN"] = hf_token
print("✔ Hugging Face token authenticated.")

# 3. System & Python Dependencies
!apt-get update -qq && apt-get install -y -qq --no-install-recommends ffmpeg libcairo2-dev libpango1.0-dev pkg-config cmake build-essential git git-lfs
!pip install -q --upgrade "transformers>=4.48.0" "peft>=0.14.0" "accelerate>=1.2.0" "safetensors>=0.5.0" "huggingface_hub>=0.28.0" "gguf>=0.10.0" sentencepiece protobuf "manim>=0.18.0" manim-voiceover

# 4. Build llama.cpp
if not (LLAMA_CPP_DIR / "convert_hf_to_gguf.py").is_file():
    !git clone --depth 1 https://github.com/ggerganov/llama.cpp.git {LLAMA_CPP_DIR}
!pip install -q -e {LLAMA_CPP_DIR}/gguf-py

build_dir = LLAMA_CPP_DIR / "build"
quant_bin = build_dir / "bin" / "llama-quantize"
cli_bin = build_dir / "bin" / "llama-cli"
if not quant_bin.is_file() or not cli_bin.is_file():
    cuda_flag = "-DGGML_CUDA=ON" if has_cuda else ""
    !cmake -B {build_dir} -S {LLAMA_CPP_DIR} -DCMAKE_BUILD_TYPE=Release {cuda_flag}
    !cmake --build {build_dir} --config Release -j{cpu_count} --target llama-quantize llama-cli

print("✔ llama.cpp binaries ready.")
!df -h {WORK_DIR}


In [ ]:
# 02_download_models
from huggingface_hub import snapshot_download

BASE_MODEL_ID = "Qwen/Qwen3-8B"
ADAPTER_MODEL_ID = "nabin2004/qwen-Manimator-1-grpo-clean"
MERGED_REPO_ID = "nabin2004/qwen-Manimator-1-grpo-merged"
GGUF_REPO_ID = "nabin2004/qwen-Manimator-1-grpo-GGUF"

adapter_cache = WORK_DIR / "adapter_cache"
print(f"Downloading LoRA adapter: {ADAPTER_MODEL_ID}...")
adapter_path = snapshot_download(repo_id=ADAPTER_MODEL_ID, local_dir=str(adapter_cache), token=hf_token)
adapter_size = sum(f.stat().st_size for f in adapter_cache.rglob('*') if f.is_file()) / 1e6
print(f"✔ Adapter downloaded: {adapter_size:.2f} MB")

print(f"Target Base Model: {BASE_MODEL_ID} (~16 GB unquantized)")
!df -h {WORK_DIR}


In [ ]:
# 03_merge_lora
import gc, torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

MERGED_DIR = WORK_DIR / "merged_model"
torch_dtype = torch.bfloat16 if (torch.cuda.is_available() and torch.cuda.is_bf16_supported()) else torch.float16
print(f"Loading base model in {torch_dtype} on CPU to preserve VRAM...")

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    torch_dtype=torch_dtype,
    device_map="cpu",
    trust_remote_code=True,
    token=hf_token,
)
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID, trust_remote_code=True, token=hf_token)

print("Applying LoRA adapter and merging weights...")
model = PeftModel.from_pretrained(base_model, ADAPTER_MODEL_ID, token=hf_token)
merged_model = model.merge_and_unload()

print(f"Saving merged weights to {MERGED_DIR}...")
MERGED_DIR.mkdir(parents=True, exist_ok=True)
merged_model.save_pretrained(MERGED_DIR, max_shard_size="4GB", safe_serialization=True)
tokenizer.save_pretrained(MERGED_DIR)

del merged_model, model, base_model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

merged_size_gb = sum(f.stat().st_size for f in MERGED_DIR.rglob('*') if f.is_file()) / 1e9
print(f"✔ Merged model saved: {merged_size_gb:.2f} GB")
!df -h {WORK_DIR}


In [ ]:
# 03b_upload_merged_model
from huggingface_hub import HfApi

api = HfApi(token=hf_token)
api.create_repo(MERGED_REPO_ID, repo_type="model", exist_ok=True, token=hf_token)

merged_readme = MERGED_DIR / "README.md"
readme_text = f"""---
license: apache-2.0
base_model: {BASE_MODEL_ID}
library_name: transformers
pipeline_tag: text-generation
tags:
  - safetensors
  - manim
  - manimce
  - grpo
  - aos
---

# {MERGED_REPO_ID}

Full-weight merged release (**Safetensors**) of **qwen-Manimator-1-grpo-clean** fine-tuned from `{BASE_MODEL_ID}` for **ManimCE** animation generation.

- **Base Model**: [`{BASE_MODEL_ID}`](https://huggingface.co/{BASE_MODEL_ID})
- **LoRA Adapter**: [`{ADAPTER_MODEL_ID}`](https://huggingface.co/{ADAPTER_MODEL_ID})
- **Merged Weights**: [`{MERGED_REPO_ID}`](https://huggingface.co/{MERGED_REPO_ID})
- **Quantized GGUF**: [`{GGUF_REPO_ID}`](https://huggingface.co/{GGUF_REPO_ID})

## Direct vLLM Deployment

```bash
python3 -m vllm.entrypoints.openai.api_server \
    --model {MERGED_REPO_ID} \
    --host 0.0.0.0 \
    --port 8000 \
    --max-model-len 32768 \
    --gpu-memory-utilization 0.90
```
"""
merged_readme.write_text(readme_text, encoding="utf-8")

print(f"Uploading merged Safetensors model folder to {MERGED_REPO_ID}...")
api.upload_folder(folder_path=str(MERGED_DIR), repo_id=MERGED_REPO_ID, repo_type="model", token=hf_token)
print(f"✔ Merged Safetensors live at https://huggingface.co/{MERGED_REPO_ID}")


In [ ]:
# 04_convert_to_gguf
F16_GGUF_PATH = WORK_DIR / "qwen-Manimator-1-grpo-f16.gguf"

print(f"Converting {MERGED_DIR} -> {F16_GGUF_PATH}...")
env = os.environ.copy()
env["PYTHONPATH"] = f"{LLAMA_CPP_DIR / 'gguf-py'}:{env.get('PYTHONPATH', '')}"

subprocess.run(
    [sys.executable, str(LLAMA_CPP_DIR / "convert_hf_to_gguf.py"), str(MERGED_DIR), "--outfile", str(F16_GGUF_PATH), "--outtype", "f16"],
    env=env,
    check=True
)

f16_size_gb = F16_GGUF_PATH.stat().st_size / 1e9
print(f"✔ F16 GGUF created: {f16_size_gb:.2f} GB")
!df -h {WORK_DIR}


In [ ]:
# 05_quantize_q8
QUANT_TYPE = "Q8_0"
Q8_GGUF_PATH = WORK_DIR / "qwen-Manimator-1-grpo-Q8_0.gguf"
quant_bin = LLAMA_CPP_DIR / "build" / "bin" / "llama-quantize"

print(f"Quantizing F16 -> {Q8_GGUF_PATH.name} (Q8_0)...")
subprocess.run([str(quant_bin), str(F16_GGUF_PATH), str(Q8_GGUF_PATH), QUANT_TYPE], check=True)

q8_size_gb = Q8_GGUF_PATH.stat().st_size / 1e9
print(f"✔ Q8_0 GGUF created: {q8_size_gb:.2f} GB")
!df -h {WORK_DIR}


In [ ]:
# 06_validate_model
from gguf import GGUFReader

reader = GGUFReader(str(Q8_GGUF_PATH))
arch = None
ctx_len = None
for field in reader.fields.values():
    if field.name == "general.architecture":
        arch = str(bytes(field.parts[field.data[0]]).decode("utf-8", errors="ignore"))
    elif "context_length" in field.name:
        try:
            ctx_len = int(field.parts[field.data[0]][0])
        except Exception:
            ctx_len = int(field.data[0])

print(f"Architecture:           {arch}")
print(f"Metadata Context Limit: {ctx_len}")

cli_bin = LLAMA_CPP_DIR / "build" / "bin" / "llama-cli"
res = subprocess.run(
    [str(cli_bin), "-m", str(Q8_GGUF_PATH), "-p", "<|im_start|>user\nSay 'GGUF Validated'\n<|im_end|>\n<|im_start|>assistant\n", "-n", "24", "--temp", "0.1", "--no-display-prompt"],
    capture_output=True,
    text=True
)
print("Validation Generation Output:", res.stdout.strip())
gguf_val_pass = (res.returncode == 0)
print(f"GGUF Validation: {'PASS' if gguf_val_pass else 'FAIL'}")


In [ ]:
# 07_test_32k_context
CONTEXT_LENGTH = 32768
print(f"Testing runtime context = {CONTEXT_LENGTH}...")
res_32k = subprocess.run(
    [str(cli_bin), "-m", str(Q8_GGUF_PATH), "-c", str(CONTEXT_LENGTH), "-p", "<|im_start|>user\nWrite a 1-sentence math insight.\n<|im_end|>\n<|im_start|>assistant\n", "-n", "32", "--temp", "0.2", "--no-display-prompt"],
    capture_output=True,
    text=True
)
print("32K Context Output:", res_32k.stdout.strip())
ctx_pass = (res_32k.returncode == 0 and len(res_32k.stdout.strip()) > 5)
print(f"32K Context Validation: {'PASS' if ctx_pass else 'FAIL'}")


In [ ]:
# 08_test_manim
import re
MANIM_OUT_DIR = WORK_DIR / "manim_artifacts"
MANIM_OUT_DIR.mkdir(parents=True, exist_ok=True)

prompt = (
    "<|im_start|>system\nYou are an expert ManimCE programmer.\n<|im_end|>\n"
    "<|im_start|>user\nWrite a complete Manim Community Edition Python script animating a blue Circle and a label 'Manimator Q8_0'. Output python inside ```python ```\n<|im_end|>\n"
    "<|im_start|>assistant\n"
)
res = subprocess.run(
    [str(cli_bin), "-m", str(Q8_GGUF_PATH), "-p", prompt, "-n", "512", "--temp", "0.2", "--no-display-prompt"],
    capture_output=True,
    text=True
)

raw_text = res.stdout.strip()
match = re.search(r"```python(.*?)```", raw_text, re.DOTALL)
code = match.group(1).strip() if match else raw_text
if "from manim import" not in code or "Scene" not in code:
    code = (
        "from manim import *\n\n"
        "class ManimatorValidation(Scene):\n"
        "    def construct(self):\n"
        "        c = Circle(color=BLUE, radius=1.5)\n"
        "        t = Text('Manimator Q8_0', font_size=36).next_to(c, DOWN)\n"
        "        self.play(Create(c))\n"
        "        self.play(Write(t))\n"
        "        self.wait(1)\n"
    )
    gen_pass = False
else:
    gen_pass = True

script_path = MANIM_OUT_DIR / "test_scene.py"
script_path.write_text(code, encoding="utf-8")

print("Rendering generated Manim script...")
render_res = subprocess.run([sys.executable, "-m", "manim", "-ql", "--media_dir", str(MANIM_OUT_DIR), str(script_path)], capture_output=True, text=True)
mp4_files = list(MANIM_OUT_DIR.rglob("*.mp4"))
render_pass = len(mp4_files) > 0 and mp4_files[0].stat().st_size > 1000
if render_pass:
    print(f"✔ Rendered video artifact saved: {mp4_files[0]} ({mp4_files[0].stat().st_size / 1e6:.2f} MB)")

# Purge intermediate F16 GGUF to free disk space
if F16_GGUF_PATH.is_file():
    F16_GGUF_PATH.unlink()
    print("✔ Cleaned up intermediate F16 GGUF.")
!df -h {WORK_DIR}


In [ ]:
# 09_upload_to_huggingface
readme_path = WORK_DIR / "README.md"
readme_content = f"""---
license: apache-2.0
base_model: {BASE_MODEL_ID}
library_name: gguf
pipeline_tag: text-generation
tags:
  - manim
  - manimce
  - gguf
  - q8_0
---

# {GGUF_REPO_ID}

High-precision **Q8_0 GGUF** release of **qwen-Manimator-1-grpo-clean** fine-tuned from `{BASE_MODEL_ID}`.

- **Base**: `{BASE_MODEL_ID}`
- **Adapter**: `{ADAPTER_MODEL_ID}`
- **Merged Model**: [`{MERGED_REPO_ID}`](https://huggingface.co/{MERGED_REPO_ID})
- **File**: `{Q8_GGUF_PATH.name}` (~{q8_size_gb:.2f} GB)
- **Context**: 32768 tokens

```bash
ollama run hf.co/{GGUF_REPO_ID}
```
"""
readme_path.write_text(readme_content, encoding="utf-8")

api.create_repo(GGUF_REPO_ID, repo_type="model", exist_ok=True, token=hf_token)
api.upload_file(path_or_fileobj=str(readme_path), path_in_repo="README.md", repo_id=GGUF_REPO_ID, repo_type="model", token=hf_token)
print(f"Uploading {Q8_GGUF_PATH.name} ({q8_size_gb:.2f} GB) to {GGUF_REPO_ID}...")
api.upload_file(path_or_fileobj=str(Q8_GGUF_PATH), path_in_repo=Q8_GGUF_PATH.name, repo_id=GGUF_REPO_ID, repo_type="model", token=hf_token)
upload_pass = True
print("✔ Hugging Face GGUF upload complete.")


In [ ]:
# 10_verify_upload
merged_files = list(api.list_repo_files(MERGED_REPO_ID, repo_type="model", token=hf_token))
gguf_files = list(api.list_repo_files(GGUF_REPO_ID, repo_type="model", token=hf_token))

merged_pass = "config.json" in merged_files and any(f.endswith(".safetensors") for f in merged_files)
gguf_pass = Q8_GGUF_PATH.name in gguf_files and "README.md" in gguf_files

print(f"""
========================================
MANIMATOR GGUF & MERGED DEPLOYMENT COMPLETE
========================================

Base:
{BASE_MODEL_ID}

Adapter:
{ADAPTER_MODEL_ID}

Merged Model (Safetensors):
https://huggingface.co/{MERGED_REPO_ID}

Merged Upload:
{'PASS' if merged_pass else 'FAIL'}

Quantization:
Q8_0

GGUF:
{Q8_GGUF_PATH.name}

Size:
{q8_size_gb:.2f} GB

Context:
32768 tokens

GGUF validation:
{'PASS' if (gguf_val_pass and ctx_pass) else 'FAIL'}

Manim generation:
{'PASS' if gen_pass else 'FAIL'}

Manim rendering:
{'PASS' if render_pass else 'FAIL'}

Hugging Face GGUF:
https://huggingface.co/{GGUF_REPO_ID}

GGUF Upload:
{'PASS' if gguf_pass else 'FAIL'}
========================================
""")
